<a href="https://colab.research.google.com/github/CokieLee/ProbML-Connections-Project/blob/ananya/GREEN_GENERATOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install nltk wordfreq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 14.3 MB/s eta 0:00:00


In [7]:
import re
import json
import random
import itertools
from collections import defaultdict, Counter

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.corpus import wordnet as wn
from wordfreq import zipf_frequency


# ---------------------------
# config
# ---------------------------

RANDOM_SEED = 7
random.seed(RANDOM_SEED)

MIN_ZIPF = 3.0
MIN_LEN = 3
MAX_LEN = 12


# ---------------------------
# label cleanup lists
# ---------------------------

# bad label words we do not want showing up as category heads
BAD_LABEL_WORDS = {
    "the", "a", "an",
    "thing", "things",
    "object", "objects",
    "entity", "entities",
    "something", "anything", "everything",
    "item", "items",
    "stuff",
    "part", "parts",
    "whole", "wholes",
    "unit", "units",
    "matter",
    "abstraction",
    "group", "groups",
    "state", "states",
    "event", "events",
    "kind", "kinds",
    "sort", "sorts",
    "class", "classes",
    "piece", "pieces"
}

# parents that are usually too abstract to make good connections categories
BAD_PARENT_LABELS = {
    "entity", "object", "thing", "whole", "part",
    "abstraction", "group", "state", "event",
    "matter", "unit", "item", "stuff"
}


# ---------------------------
# basic word helpers
# ---------------------------

def normalize_word(w: str):
    # normalize word form for comparisons and dedupe
    return w.lower().strip()


def pretty_word(w: str):
    # display version of a word in all caps
    return w.upper()

def pretty_group(words):
    # display version of a word group in all caps
    return tuple(pretty_word(w) for w in words)

def pretty_label(label: str):
    # display version of a category label in all caps
    return label.upper()


def is_good_word(w: str, min_zipf=MIN_ZIPF):
    # keep only simple single-token english words that are reasonably common
    if not isinstance(w, str):
        return False

    if "_" in w or "-" in w or " " in w:
        return False

    if not re.fullmatch(r"[A-Za-z]+", w):
        return False

    if len(w) < MIN_LEN or len(w) > MAX_LEN:
        return False

    if zipf_frequency(w.lower(), "en") < min_zipf:
        return False

    return True


def canonical_group(words):
    # canonical tuple for deduping 4-word groups regardless of order/case
    return tuple(sorted(set(normalize_word(w) for w in words)))


def score_group(words):
    # crude quality score: average word frequency
    vals = [zipf_frequency(w.lower(), "en") for w in words]
    return sum(vals) / len(vals)


def pick_best_lemma_names(synset, min_zipf=MIN_ZIPF):
    # extract clean usable lemma names from a synset
    lemmas = [l.name() for l in synset.lemmas()]
    lemmas = [w for w in lemmas if is_good_word(w, min_zipf=min_zipf)]
    lemmas = sorted(set(lemmas), key=lambda x: (-zipf_frequency(x.lower(), "en"), x.lower()))
    return lemmas


def pick_label_word(synset):
    # choose a human-usable label head for a synset and avoid junk like "the"
    candidates = pick_best_lemma_names(synset, min_zipf=2.5)
    candidates = [w for w in candidates if w.lower() not in BAD_LABEL_WORDS]

    if candidates:
        return candidates[0].lower()

    # fallback: mine a decent word from the definition
    words = re.findall(r"[A-Za-z]+", synset.definition().lower())
    for w in words:
        if (
            w not in BAD_LABEL_WORDS
            and len(w) > 3
            and zipf_frequency(w, "en") >= 2.5
        ):
            return w

    return None


def sample_groups(words, max_groups=8):
    # generate multiple distinct 4-word groups from a candidate word list
    words = list(dict.fromkeys(words))

    if len(words) < 4:
        return []

    if len(words) == 4:
        return [tuple(sorted(words))]

    groups = set()

    # exact combinations when pool is small enough
    if len(words) <= 10:
        combos = list(itertools.combinations(sorted(words), 4))
        random.shuffle(combos)
        return [tuple(c) for c in combos[:max_groups]]

    # otherwise random sample many times
    tries = 0
    max_tries = max_groups * 50

    while len(groups) < max_groups and tries < max_tries:
        g = tuple(sorted(random.sample(words, 4)))
        groups.add(g)
        tries += 1

    return list(groups)


# ---------------------------
# manual category banks
# ---------------------------

MANUAL_CATEGORIES = {
    "programming languages": [
        "python", "java", "ruby", "scala", "swift", "kotlin", "fortran", "cobol", "lisp", "pascal"
    ],
    "musical instruments": [
        "piano", "violin", "cello", "trumpet", "flute", "clarinet", "trombone", "harp", "banjo", "drum"
    ],
    "dog breeds": [
        "beagle", "poodle", "boxer", "collie", "husky", "terrier", "mastiff", "bulldog", "whippet", "spaniel"
    ],
    "car parts": [
        "axle", "brake", "clutch", "bumper", "mirror", "engine", "piston", "radiator", "tire", "fender"
    ],
    "fruits": [
        "apple", "pear", "peach", "plum", "grape", "melon", "guava", "papaya", "banana", "orange"
    ],
    "vegetables": [
        "onion", "carrot", "turnip", "radish", "spinach", "cabbage", "pepper", "garlic", "tomato", "celery"
    ],
    "birds": [
        "eagle", "robin", "sparrow", "falcon", "heron", "raven", "finch", "parrot", "pigeon", "owl"
    ],
    "fish": [
        "trout", "salmon", "tuna", "carp", "perch", "herring", "sardine", "catfish", "grouper", "snapper"
    ],
    "gemstones": [
        "ruby", "jade", "amber", "opal", "pearl", "topaz", "garnet", "onyx", "agate", "diamond"
    ],
    "fabrics": [
        "cotton", "silk", "linen", "denim", "velvet", "satin", "wool", "tweed", "nylon", "rayon"
    ],
    "weather words": [
        "rain", "snow", "sleet", "wind", "storm", "cloud", "frost", "thunder", "drizzle", "mist"
    ],
    "school subjects": [
        "algebra", "history", "biology", "physics", "chemistry", "geography", "grammar", "logic"
    ],
    "sports": [
        "tennis", "soccer", "cricket", "hockey", "rugby", "boxing", "golf", "squash"
    ],
    "kitchen tools": [
        "ladle", "whisk", "tongs", "grater", "peeler", "spatula", "sieve", "colander", "skillet", "kettle"
    ],
    "board games": [
        "chess", "checkers", "sorry", "risk", "scrabble", "clue", "monopoly", "dominoes"
    ],
    "flowers": [
        "rose", "lily", "tulip", "daisy", "violet", "iris", "poppy", "lotus", "orchid", "peony"
    ],
    "office supplies": [
        "stapler", "folder", "pencil", "eraser", "marker", "binder", "notebook", "highlighter"
    ],
    "clothing": [
        "shirt", "skirt", "jacket", "blazer", "sweater", "hoodie", "jeans", "blouse"
    ],
    "currencies": [
        "dollar", "euro", "peso", "rupee", "yen", "franc", "pound", "dinar"
    ],
    "sea creatures": [
        "shark", "whale", "squid", "otter", "seal", "urchin", "clam", "lobster"
    ],
}


COLOR_WORDS = sorted({
    "red", "blue", "green", "yellow", "orange", "purple", "violet", "indigo", "pink", "brown", "black",
    "white", "gray", "grey", "cyan", "teal", "navy", "maroon", "magenta", "lavender", "beige", "tan",
    "cream", "ivory", "turquoise", "aqua", "scarlet", "crimson", "ruby", "emerald", "sapphire",
    "amber", "gold", "silver", "bronze", "peach", "coral", "mint", "olive"
})


def generate_manual_categories(max_groups_per_label=12):
    # build groups from handpicked category banks that feel more connections-like
    cats = []

    for label, words in MANUAL_CATEGORIES.items():
        usable = [w for w in words if is_good_word(w, min_zipf=2.5)]

        for g in sample_groups(usable, max_groups=max_groups_per_label):
            cats.append({
                "type": "manual",
                "label": label,
                "words": g,
                "score": score_group(g)
            })

    return cats


def generate_color_categories(n_samples=500):
    # generate color-based green/easy-style categories
    usable = [c for c in COLOR_WORDS if is_good_word(c, min_zipf=2.5)]
    cats = []
    seen = set()

    for _ in range(n_samples * 4):
        g = tuple(sorted(random.sample(usable, 4)))
        if g in seen:
            continue
        seen.add(g)

        cats.append({
            "type": "colors",
            "label": "colors",
            "words": g,
            "score": score_group(g)
        })

        if len(cats) >= n_samples:
            break

    return cats


# ---------------------------
# pattern-style generators
# ---------------------------

def collect_common_wordnet_words(min_zipf=3.0):
    # collect a broad set of clean common words from wordnet for pattern mining
    out = set()

    for syn in wn.all_synsets():
        for lemma in syn.lemmas():
            w = lemma.name().lower()
            if is_good_word(w, min_zipf=min_zipf):
                out.add(w)

    return out


def generate_suffix_categories(max_categories=1200, min_bucket_size=6, groups_per_suffix=5):
    # build categories from common endings like -er or -ing
    common_words = collect_common_wordnet_words(min_zipf=3.0)
    suffixes = ["er", "or", "ist", "ing", "man", "ion", "ity", "dom", "ship", "hood"]
    buckets = defaultdict(list)

    for w in common_words:
        for s in suffixes:
            if w.endswith(s):
                buckets[s].append(w)

    cats = []

    for sfx, words in buckets.items():
        words = sorted(set(words), key=lambda x: (-zipf_frequency(x, "en"), x))

        if len(words) < min_bucket_size:
            continue

        for g in sample_groups(words, max_groups=groups_per_suffix):
            cats.append({
                "type": "suffix",
                "label": f"words ending in -{sfx}",
                "words": g,
                "score": score_group(g)
            })

            if len(cats) >= max_categories:
                return cats

    return cats


# ---------------------------
# wordnet generators
# ---------------------------

def generate_synonym_categories(max_categories=6000, groups_per_synset=6):
    # build groups from synsets that have enough clean lemma names
    cats = []

    for syn in wn.all_synsets():
        lemmas = pick_best_lemma_names(syn, min_zipf=MIN_ZIPF)

        if len(lemmas) >= 4:
            for g in sample_groups(lemmas, max_groups=groups_per_synset):
                cats.append({
                    "type": "synonyms",
                    "label": syn.definition()[:80],
                    "words": g,
                    "synset": syn.name(),
                    "score": score_group(g)
                })

                if len(cats) >= max_categories:
                    return cats

    return cats


def generate_typeof_categories(max_categories=5000, min_hyponyms=6, groups_per_parent=5, min_depth=4):
    # build "types of x" groups from parent noun synsets and their hyponyms
    # min_depth helps avoid super abstract parents like entity/object
    cats = []
    noun_synsets = list(wn.all_synsets(pos=wn.NOUN))

    for parent in noun_synsets:
        if parent.max_depth() < min_depth:
            continue

        hypos = set(parent.hyponyms())
        for h in list(hypos):
            hypos.update(h.hyponyms())

        words = []
        for h in hypos:
            lemmas = pick_best_lemma_names(h, min_zipf=MIN_ZIPF)
            if lemmas:
                words.append(lemmas[0])

        words = sorted(set(words), key=lambda x: (-zipf_frequency(x.lower(), "en"), x.lower()))

        if len(words) < min_hyponyms:
            continue

        label_head = pick_label_word(parent)
        if not label_head:
            continue

        if label_head in BAD_PARENT_LABELS:
            continue

        for g in sample_groups(words, max_groups=groups_per_parent):
            cats.append({
                "type": "types_of",
                "label": f"types of {label_head}",
                "words": g,
                "parent_synset": parent.name(),
                "score": score_group(g)
            })

            if len(cats) >= max_categories:
                return cats

    return cats


def generate_sister_categories(max_categories=3500, groups_per_parent=5, min_depth=4):
    # build sibling categories from direct child hyponyms under the same parent
    cats = []
    noun_synsets = list(wn.all_synsets(pos=wn.NOUN))

    for parent in noun_synsets:
        if parent.max_depth() < min_depth:
            continue

        children = parent.hyponyms()
        child_words = []

        for child in children:
            lemmas = pick_best_lemma_names(child, min_zipf=MIN_ZIPF)
            if lemmas:
                child_words.append(lemmas[0])

        child_words = sorted(set(child_words), key=lambda x: (-zipf_frequency(x.lower(), "en"), x.lower()))

        if len(child_words) < 4:
            continue

        label_head = pick_label_word(parent)
        if not label_head:
            continue

        if label_head in BAD_PARENT_LABELS:
            continue

        for g in sample_groups(child_words, max_groups=groups_per_parent):
            cats.append({
                "type": "sister_hyponyms",
                "label": f"siblings under {label_head}",
                "words": g,
                "parent_synset": parent.name(),
                "score": score_group(g)
            })

            if len(cats) >= max_categories:
                return cats

    return cats


def generate_part_categories(max_categories=2500, groups_per_whole=4, min_depth=4):
    # build "parts of x" groups using meronyms of a whole synset
    cats = []
    noun_synsets = list(wn.all_synsets(pos=wn.NOUN))

    for whole in noun_synsets:
        if whole.max_depth() < min_depth:
            continue

        parts = set(whole.part_meronyms()) | set(whole.substance_meronyms()) | set(whole.member_meronyms())
        words = []

        for p in parts:
            lemmas = pick_best_lemma_names(p, min_zipf=MIN_ZIPF)
            if lemmas:
                words.append(lemmas[0])

        words = sorted(set(words), key=lambda x: (-zipf_frequency(x.lower(), "en"), x.lower()))

        if len(words) < 4:
            continue

        label_head = pick_label_word(whole)
        if not label_head:
            continue

        if label_head in BAD_PARENT_LABELS:
            continue

        for g in sample_groups(words, max_groups=groups_per_whole):
            cats.append({
                "type": "parts_of",
                "label": f"parts of {label_head}",
                "words": g,
                "parent_synset": whole.name(),
                "score": score_group(g)
            })

            if len(cats) >= max_categories:
                return cats

    return cats


# ---------------------------
# dedupe / filtering / ranking
# ---------------------------

def dedupe_categories(categories):
    # remove exact duplicate category type + label + word-group combinations
    seen = set()
    out = []

    for c in categories:
        key = (c["type"], c["label"].lower(), canonical_group(c["words"]))
        if key not in seen:
            seen.add(key)
            out.append(c)

    return out


def filter_categories(categories, min_avg_zipf=3.2):
    # remove weak or malformed categories after generation
    out = []

    for c in categories:
        words = c["words"]

        if len(set(map(str.lower, words))) != 4:
            continue

        if score_group(words) < min_avg_zipf:
            continue

        out.append(c)

    return out


def polysemy_count(w):
    # count how ambiguous a word is in wordnet
    return len(wn.synsets(w.lower()))


def lemma_neighbors(w):
    # collect nearby semantic neighbors for cheap collision checks
    out = set()

    for syn in wn.synsets(w.lower()):
        for l in syn.lemmas():
            name = l.name().lower()
            if is_good_word(name, min_zipf=2.0):
                out.add(name)

        for rel in syn.hypernyms() + syn.hyponyms():
            for l in rel.lemmas():
                name = l.name().lower()
                if is_good_word(name, min_zipf=2.0):
                    out.add(name)

    return out


def categories_too_related(cat1, cat2):
    # reject category pairs that are likely to muddy the puzzle
    s1 = set(map(str.lower, cat1["words"]))
    s2 = set(map(str.lower, cat2["words"]))

    if s1 & s2:
        return True

    neigh1 = set()
    neigh2 = set()

    for w in s1:
        neigh1 |= lemma_neighbors(w)
    for w in s2:
        neigh2 |= lemma_neighbors(w)

    overlap = len(neigh1 & neigh2)
    return overlap >= 3


def rank_categories(categories):
    # sort categories so stronger/more human-readable ones tend to float upward
    def quality_key(c):
        label_bonus = 0.0

        if c["type"] in {"manual", "colors"}:
            label_bonus += 0.25

        if c["type"] in {"types_of", "sister_hyponyms", "parts_of"}:
            label_bonus += 0.10

        return (
            -(c.get("score", 0.0) + label_bonus),
            c["type"],
            c["label"],
        )

    return sorted(categories, key=quality_key)


# ---------------------------
# puzzle generation
# ---------------------------

def generate_puzzle(category_bank, max_tries=50000, max_polysemy=8, enforce_distinct_labels=True):
    # sample 4 categories that do not collide too badly and produce 16 distinct words
    bank = category_bank[:]

    for _ in range(max_tries):
        chosen = random.sample(bank, 4)

        if enforce_distinct_labels:
            labels = [c["label"] for c in chosen]
            if len(set(labels)) < 4:
                continue

        bad = False
        for a, b in itertools.combinations(chosen, 2):
            if categories_too_related(a, b):
                bad = True
                break

        if bad:
            continue

        words = [w for cat in chosen for w in cat["words"]]

        if len(set(map(str.lower, words))) != 16:
            continue

        if any(polysemy_count(w) > max_polysemy for w in words):
            continue

        shuffled = words[:]
        random.shuffle(shuffled)
        return chosen, shuffled

    return None, None


# ---------------------------
# full bank builder
# ---------------------------

def build_category_bank(
    synonym_max=7000,
    typeof_max=6000,
    sister_max=4000,
    part_max=3000,
    color_n=600,
    manual_groups=16,
    suffix_max=1000,
    keep_top_n=15000,
):
    # build the full category bank from multiple sources, then clean and rank it
    syn_cats = generate_synonym_categories(
        max_categories=synonym_max,
        groups_per_synset=6,
    )

    type_cats = generate_typeof_categories(
        max_categories=typeof_max,
        min_hyponyms=6,
        groups_per_parent=5,
        min_depth=4,
    )

    sister_cats = generate_sister_categories(
        max_categories=sister_max,
        groups_per_parent=5,
        min_depth=4,
    )

    part_cats = generate_part_categories(
        max_categories=part_max,
        groups_per_whole=4,
        min_depth=4,
    )

    color_cats = generate_color_categories(
        n_samples=color_n,
    )

    manual_cats = generate_manual_categories(
        max_groups_per_label=manual_groups,
    )

    suffix_cats = generate_suffix_categories(
        max_categories=suffix_max,
        min_bucket_size=6,
        groups_per_suffix=5,
    )

    category_bank = (
        syn_cats
        + type_cats
        + sister_cats
        + part_cats
        + color_cats
        + manual_cats
        + suffix_cats
    )

    category_bank = dedupe_categories(category_bank)
    category_bank = filter_categories(category_bank, min_avg_zipf=3.2)
    category_bank = rank_categories(category_bank)

    if keep_top_n is not None:
        category_bank = category_bank[:keep_top_n]

    return category_bank


# ---------------------------
# display / export helpers
# ---------------------------

def print_sample_categories(category_bank, k=10):
    # print a small random sample so you can inspect bank quality
    print(f"Category bank size: {len(category_bank)}")
    print("\nExamples:")

    if not category_bank:
        return

    for c in random.sample(category_bank, min(k, len(category_bank))):
        print(f'{c["type"]:16} | {pretty_label(c["label"])[:55]:55} | {pretty_group(c["words"])}')


def summarize_bank(category_bank):
    # show counts by category type to understand composition of the bank
    counts = Counter(c["type"] for c in category_bank)
    print("\nCounts by type:")
    for k, v in counts.most_common():
        print(f"{k:18} {v}")


def print_puzzle(chosen_cats, puzzle_words):
    # display one generated puzzle and its hidden categories
    if not chosen_cats:
        print("No puzzle found.")
        return

    print("Puzzle words:")
    print([pretty_word(w) for w in puzzle_words])
    print("\nCategories:")

    for i, c in enumerate(chosen_cats, 1):
        print(f'{i}. {pretty_label(c["label"])} -> {pretty_group(c["words"])} [{c["type"]}]')


def make_display_bank(category_bank):
    # create a copy of the bank with uppercase labels and words for export/display
    out = []

    for c in category_bank:
        c2 = dict(c)
        c2["label"] = pretty_label(c["label"])
        c2["words"] = [pretty_word(w) for w in c["words"]]
        out.append(c2)

    return out


def export_category_bank_json(category_bank, path="category_bank.json"):
    # export full bank to json for reuse or later analysis
    export_bank = make_display_bank(category_bank)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(export_bank, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(export_bank)} categories to {path}")


# ---------------------------
# run
# ---------------------------



In [8]:
category_bank = build_category_bank(
        synonym_max=7000,
        typeof_max=6000,
        sister_max=4000,
        part_max=3000,
        color_n=600,
        manual_groups=16,
        suffix_max=1000,
        keep_top_n=15000,
    )

print_sample_categories(category_bank, k=12)
summarize_bank(category_bank)

chosen_cats, puzzle_words = generate_puzzle(category_bank,
        max_tries=50000,
        max_polysemy=8,
        enforce_distinct_labels=True,
    )

print()
print_puzzle(chosen_cats, puzzle_words)

# optional export
# export_category_bank_json(category_bank, "category_bank.json")

Category bank size: 13923

Examples:
types_of         | TYPES OF CLEANING                                       | ('BATHING', 'SANITATION', 'SCRUB', 'SWEEPING')
synonyms         | SHOCKINGLY REPELLENT; INSPIRING HORROR                  | ('GHASTLY', 'GRIM', 'GRUESOME', 'SICK')
types_of         | TYPES OF EQUIPMENT                                      | ('BOOSTER', 'DETECTOR', 'PHONE', 'SCOPE')
types_of         | TYPES OF EVIL                                           | ('CRUELTY', 'MALICE', 'SPITE', 'VICE')
types_of         | TYPES OF VALUE                                          | ('GREATNESS', 'RICHNESS', 'SIGNIFICANCE', 'URGENCY')
types_of         | TYPES OF CREATING                                       | ('ERECTION', 'GRADING', 'MANUFACTURE', 'SHIPBUILDING')
synonyms         | REQUIRE AS USEFUL, JUST, OR PROPER                      | ('ASK', 'INVOLVE', 'REQUIRE', 'TAKE')
sister_hyponyms  | SIBLINGS UNDER RODENT                                   | ('BEAVER', 'HAMSTER', 'MARA', 'RA

In [10]:
print_sample_categories(category_bank, k=40)


Category bank size: 13923

Examples:
sister_hyponyms  | SIBLINGS UNDER CREATION                                 | ('COPY', 'DOCUMENT', 'ILLUSTRATION', 'SHADE')
sister_hyponyms  | SIBLINGS UNDER JOURNEY                                  | ('DRIVE', 'ODYSSEY', 'PASSAGE', 'TREK')
types_of         | TYPES OF DEALING                                        | ('CONVEYING', 'DISTRIBUTION', 'TRADING', 'TRAFFIC')
synonyms         | A VERY ATTRACTIVE OR SEDUCTIVE LOOKING WOMAN            | ('DISH', 'KNOCKOUT', 'PEACH', 'SWEETHEART')
sister_hyponyms  | SIBLINGS UNDER GETTING                                  | ('ACQUISITION', 'CAPTURE', 'CATCHING', 'OCCUPATION')
types_of         | TYPES OF HARD                                           | ('ALMOND', 'CHESTNUT', 'COCONUT', 'PEANUT')
types_of         | TYPES OF AGREEMENT                                      | ('COMPACT', 'COVENANT', 'OBLIGATION', 'SUBMISSION')
types_of         | TYPES OF GAME                                           | ('MONOPOLY', 'SC